# Desplegar modelo Churn Client
Despliegue de caracteristicas y modelo para realizar predicciones en tiempo real via llamdas REST API.
Como las predicciones son realizadas en el front de aplicaciones cliente la respuesta debe ser de baja latencia y alta concurrencia.

Servir caracteristicas y modelo:
- Caracteristicas disponibles con baja latencia a traves de Databricks tablas online.
- Desplegar el modelo registrado en UC a un endpoint servicio de modelo de baja latencia.

## 1. Configuración

In [0]:
import sys
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Versión de Python actual: {python_version}")

Versión de Python actual: 3.12.3


In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("model_version", "1", "Modelo Version (@champion)")

In [0]:
model_full_name = dbutils.widgets.get("model_full_name")
model_version = int(dbutils.widgets.get("model_version")) 
tag_name = dbutils.widgets.get("approval_tag_name") 
catalog =  model_full_name.split('.')[0]
schema =  model_full_name.split('.')[1]
model_name =  model_full_name.split('.')[-1]
model_alias_challenger = "challenger" 
model_alias_champion = "champion"
model_alias_production = "production"
delete_other_aliases = True

endpoint_name = (f"{model_name}_endpoint") # (f"{model_name}_endpoint")[:50]
wait_time_minutes = 20 # La ultima ejecución demoro 10 minutos
promote_model = True

## 2. Desplegar el modelo para inferencia en tiempo real
Para realizar predicciones en tiempo real via llamdas REST API, desplegamos a un endpoint servicio de modelo. Aplicaciones cliente pueden usar el endpoint. El servicio maneja toda la infraestructura, despliegue y escalado.

## Promover el modelo para servicio en tiempo real
Anteriormente usamos los alias `@challenger` y `@champion` para promover modelo para ser llamado en un pipeline batch.
Como inferencia en tiempo real es otra forma de consumir el modelo, necesitamos otro flujo de trabajo para desplegar el modelo, existen varias estrategias como A/B pruebas, despliegue canario, prueba sombra, etc.

En pruebas A/B, hay un modelo vivo en producción. Cuando tenemos un nuevo modelo queremos desviar un porcentaje del trafico en línea a la nueva versión. Por ejemplo, 80% al modelo `production` y 20% al `champion`. De esta forma comparamos los resultados de los dos modelos. Si los resultados son aceptables desviamos todo el trafico a la nueva versión para convertirse en la versión viva.

Introducimos un nuevo alias, `@production`, para rastrear este flujo:
- `@production`: Modelo production que está 'vivo'.
- `@champion`: Nueva versión que fue promovida despues de las pruebas champion-challenger.
- `@challenger`: Modelo que desafía al campeon. Nunga desplegar para servicio en tiempo real a menos que sea promovido a champion.

### Promover modelo

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

def get_last_model_version(model_name):
    versions = client.search_model_versions(f"name='{model_name}'")
    latest = max(versions, key=lambda v: int(v.version))
    return latest

#model_details = get_last_model_version(model_full_name)
model_details = client.get_model_version(model_full_name, model_version)

if not model_details:
    raise Exception("No encontrado version del modelo registrado en UC.")

tags = model_details.tags
aliases = model_details.aliases

# Check if any tag matches the approval tag name
if not any(tag == tag_name for tag in tags.keys()):
  raise Exception("Version del modelo no aprobada para despliegue")
else:
  if tags.get(tag_name).lower() == "approved":
    print("Version del modelo aprobada para despliegue")
    
    if promote_model:
      if model_alias_production.lower() in [alias.lower() for alias in aliases]:
        print(f"El modelo ya tiene el alias: {model_alias_production}. Ya fue promovido a esta etapa.")
      else:
        print(f"El modelo No tiene el alias: {model_alias_production}. Será promovido a esta etapa!")
        
        if delete_other_aliases:
          try:
            for alias in aliases:
                if alias.lower() != model_alias_production.lower():
                    client.delete_registered_model_alias(name=model_full_name, alias=alias)
          except:
            pass

        client.set_registered_model_alias(name=model_full_name, alias=model_alias_production, version=model_details.version)
  else:
    raise Exception("Version del modelo no aprobada para despliegue")

Version del modelo aprobada para despliegue
El modelo No tiene el alias: production. Será promovido a esta etapa!


## 3. Crear endpoint servicio de modelo

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceDoesNotExist
import time

def exists_endpoint(endpoint_name):
    """Verifica si existe un serving endpoint.
    
    Args:
        endpoint_name: Nombre del endpoint a verificar
        
    Returns:
        True si el endpoint existe, False en caso contrario
    """
    try:
        w = WorkspaceClient()
        w.serving_endpoints.get(endpoint_name)
        return True
    except ResourceDoesNotExist:
        return False
    except Exception as e:
        print(f"Error al verificar endpoint: {e}")
        return False

def wait_until_endpoint_ready(endpoint_name, timeout=900, sleep_time=30):
    """Espera hasta que el endpoint esté listo para servir.
    
    Args:
        endpoint_name: Nombre del endpoint
        timeout: Tiempo máximo de espera en segundos (default: 900s = 15 min)
        sleep_time: Tiempo de espera entre chequeos en segundos (default: 30s)
        
    Returns:
        True si el endpoint está listo, False si se agota el tiempo
    """
    w = WorkspaceClient()
    start_time = time.time()
    
    while time.time() - start_time < timeout:
        try:
            endpoint = w.serving_endpoints.get(endpoint_name)
            
            # Verificar estado del endpoint
            if (endpoint.state and 
                endpoint.state.ready and 
                endpoint.state.ready.value == "READY" and
                endpoint.state.config_update and
                endpoint.state.config_update.value == "NOT_UPDATING"):
                return True
            
            print(f"Endpoint {endpoint_name} aún no está listo. Estado: {endpoint.state.ready.value if endpoint.state and endpoint.state.ready else 'UNKNOWN'}. Esperando...")
            
        except Exception as e:
            print(f"Error al verificar estado del endpoint: {e}")
        
        time.sleep(sleep_time)
    
    return False

In [0]:
from databricks.sdk.service.serving import EndpointCoreConfigInput

served_model_name =  model_name
endpoint_config_dict = {
    "served_entities": [
        # Add models to be served to this list
        {
            "entity_name": model_full_name,
            "entity_version": model_version,
            "scale_to_zero_enabled": True,
            "workload_size": "Small",
        },
        # {
        #     "entity_name": model_full_name,
        #     "entity_version": model_version_b,
        #     "scale_to_zero_enabled": True,
        #     "workload_size": "Small",
        # },
    ],
    "traffic_config": {
        "routes": [
            # Add versions of the model to be served to this list
            # Make sure that traffic_percentage adds up to 100 over all served models
            # Naming convention for served_model_name: <registered_model_name>-<model_version>
            {"served_model_name": f"{served_model_name}-{model_version}", "traffic_percentage": 100},
            # {"served_model_name": f"{served_model_name}-{model_version_b}", "traffic_percentage": 10},
        ]
    },
    # Legacy inference tables han sido deprecadas
    # Usar AI Gateway inference tables en su lugar o deshabilitar auto_capture_config
    # "auto_capture_config":{"catalog_name": catalog, "schema_name": schema, "table_name_prefix": "advanced_churn_served", "enabled": False}
}

endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)

In [0]:
from databricks.sdk.service.serving import EndpointTag
from databricks.sdk.errors import ResourceDoesNotExist
from databricks.sdk import WorkspaceClient

try:
  w = WorkspaceClient()

  if exists_endpoint(endpoint_name):
    w.serving_endpoints.update_config(name=endpoint_name, config=endpoint_config,
        served_entities=endpoint_config.served_entities,
        traffic_config=endpoint_config.traffic_config
      )
    print(f"Actualizado endpoint: {endpoint_name}, con modelo: {model_full_name}, version: {model_version}")
  else:
    w.serving_endpoints.create(name=endpoint_name, config=endpoint_config)
    #  tags=[EndpointTag.from_dict({"key": "dbdemos", "value": "advanced_mlops_churn"})]
    print(f"Creando endpoint: {endpoint_name}, con modelo: {model_name}, version: {model_version}")    
except Exception as e:
  print(f"Error al crear o actualizar endpoint. Mensaje: {e}")
  raise e

Creando endpoint: customer_churn_classifier_endpoint, con modelo: customer_churn_classifier, version: 1


### 4. Esperar/verificar que el endpoint este listo
La ejecución toma varios minutosa para para que el endpoint este listo. Tome una pausas y cheque luego. En la version gratuita con cluster Serverless demora alrededor de 10 minutos.

In [0]:
if not exists_endpoint(endpoint_name):
  raise Exception(f"Endpoint '{endpoint_name}' no existe. Crear el endpoint primero.")

print(f"Esperando a que el endpoint '{endpoint_name}' esté listo...")  
if wait_until_endpoint_ready(endpoint_name, timeout=wait_time_minutes*60, sleep_time=30):
  print(f"✓ Endpoint '{endpoint_name}' confirmado listo — seguro enviar inferencias o peticiones.")
else:
  raise TimeoutError(f"Endpoint '{endpoint_name}' todavía no está listo después de {wait_time_minutes} minutos.")

Esperando a que el endpoint 'customer_churn_classifier_endpoint' esté listo...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Estado: NOT_READY. Esperando...
Endpoint customer_churn_classifier_endpoint aún no está listo. Est